In [8]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [38]:
import torch

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def encode(s):
    return [stoi[c] for c in s]


def decode(l):
    return ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)

In [39]:
seq_len = 50


def get_batch():
    i = torch.randint(len(data) - seq_len - 1, (1,))
    x = data[i:i + seq_len]
    y = data[i + 1:i + seq_len + 1]
    return x.unsqueeze(0), y.unsqueeze(0)

In [40]:
import torch.nn as nn


class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        x = self.embed(x)
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h

In [41]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        x = self.embed(x)
        out, h = self.lstm(x, h)
        out = self.fc(out)
        return out, h

In [42]:
def train(model, epochs=2000):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    criterion = nn.CrossEntropyLoss()

    losses = []

    for epoch in range(epochs):
        x, y = get_batch()

        logits, _ = model(x)

        loss = criterion(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 200 == 0:
            print(f"step {epoch:4}, loss {loss.item():.4f}")

        losses.append(loss.item())

    return losses

In [53]:
def generate(model, start="To be", length=200, temperature=1):
    model.eval()

    x = torch.tensor([encode(start)], dtype=torch.long)
    h = None

    result = list(start)

    for _ in range(length):
        logits, h = model(x, h)

        # no temp
        # probs = torch.softmax(logits, dim=-1)

        # temp
        logits = logits[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)

        idx = torch.multinomial(probs, num_samples=1).item()

        result.append(itos[idx])

        x = torch.tensor([[idx]])

    return ''.join(result)

In [44]:
# RNN
rnn_model = CharRNN(vocab_size, 128)

In [54]:
rnn_losses  = train(rnn_model)

step    0, loss 2.1084
step  200, loss 1.8316
step  400, loss 1.8794
step  600, loss 1.9936
step  800, loss 1.7385
step 1000, loss 1.8666
step 1200, loss 1.8315
step 1400, loss 1.8600
step 1600, loss 2.0645
step 1800, loss 2.0139


In [55]:
print(generate(rnn_model))

To be not do sumap thou to and tyu him you be circourmes of soury ye sersen thou hime Tur his have that mes nath. Aring. Gright ylubs these welled.
Go?
Sime, to hand:
Marddesen:
But hat thy leser should, w


In [56]:
# LSTM
lstm_model = CharLSTM(vocab_size, 128)

In [57]:
lstm_losses = train(lstm_model)

step    0, loss 4.1840
step  200, loss 2.1758
step  400, loss 2.1223
step  600, loss 2.1736
step  800, loss 1.8617
step 1000, loss 1.9237
step 1200, loss 2.0409
step 1400, loss 1.8776
step 1600, loss 2.1612
step 1800, loss 1.9718


In [58]:
print(generate(lstm_model))

To best breed mifeaty amer andwith thou Lords,
Thitn, vere
Than mady rath alk!

WARICHAEIO:
Thans, I such and the show is thou king the recy chat kings didstere is this fainty, angard ance ore e'sly.

GCOR


In [ ]:
import matplotlib.pyplot as plt

plt.plot(rnn_losses, label="RNN")
plt.plot(lstm_losses, label="LSTM")
plt.legend()
plt.title("Loss comparison")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()